# Identyfikacja baniek spekulacyjnych na rynku kryptowalut

Notebook prowadzi przez cztery etapy pipeline'u: pobranie danych, przygotowanie, modele
zmienności i test PSY. Logiki obliczeniowej tu nie ma - ta mieszka w `src/`. Każda komórka kodu
wywołuje jeden etap i wypisuje jego wynik, więc przebieg od góry do dołu odtwarza wszystkie pliki
w `data/` i `reports/`.

Etap czwarty wymaga karty NVIDIA. Pomiar na GeForce RTX 4060 Ti: przygotowanie danych i modele
zmienności zajmują 1 minutę 46 sekund, test PSY 10 minut 44 sekundy.

In [ ]:
from pathlib import Path
import sys
import warnings

# Korzeń projektu wyznaczany przez szukanie w górę, bo katalog roboczy notebooka
# zależy od edytora. Wydruk od razu pokazuje, czy środowisko wystartowało tam, gdzie trzeba.
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'src' / 'data_load.py').is_file():
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        raise RuntimeError(f'Nie znaleziono korzenia projektu startując z {Path.cwd()}.')
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('PROJECT_ROOT:', PROJECT_ROOT)

In [ ]:
warnings.filterwarnings('ignore')

from src import data_load, model, paths, pre_processing, validation
from src.visualisation import plot_garch_decomposition, set_figures_dir

paths.ensure_directories()
print(f'szeregow: {len(data_load.CRYPTO_LIST)}')
print(f'zakres proby: {data_load.START_DATE} - {data_load.END_DATE}')
print(f'karta CUDA dostepna: {validation.gpu_available()}')

## Dane

Dzienne ceny zamknięcia 12 kryptowalut z Yahoo Finance, od 2018-01-01 do 2025-12-30. Dolną granicę
wyznacza dostępność notowań: Yahoo Finance podaje 10 z 12 szeregów dopiero od 2017-11-09, więc 2018
to pierwszy pełny rok z kompletem danych.

Pobranie uruchamia się tylko wtedy, gdy `data/raw/` jest puste, więc powtórny przebieg notebooka
nie sięga do sieci i nie podmienia danych, na których policzono wyniki.

In [ ]:
missing = [n for n in data_load.CRYPTO_LIST
           if not (paths.DATA_RAW / f'{n}.csv').is_file()]

if missing:
    print(f'brakuje {len(missing)} szeregow, pobieram z Yahoo Finance')
    data_load.download_prices()
else:
    print('data/raw/ kompletne, pobieranie pominiete')

raw = data_load.load_raw('Bitcoin')
print(f"\nBitcoin: {len(raw)} obserwacji, "
      f"{raw['Date'].min().date()} - {raw['Date'].max().date()}")
raw.head()

## Warunkowa wartość oczekiwana i trend zmienności

Ceny zamknięcia przeliczono na logarytmiczne stopy zwrotu i dopasowano do nich model AR(p), którego
rząd wybiera kryterium BIC. Reszty tego modelu rozkładają się na składnik deterministyczny
`g(t/T)` i stochastyczny `h_t`.

Składnik `g(t/T)` oszacowano lokalną jądrową regresją liniową na kwadratach reszt, z jądrem
gaussowskim i szerokością pasma dobraną walidacją krzyżową najmniejszych kwadratów. Obserwowalne są
kwadraty reszt, więc regresja estymuje lokalną wariancję, a `g(t/T)` jest jej pierwiastkiem.

In [ ]:
diagnostics = pre_processing.run()
diagnostics

Ujemne szerokości pasma dla Bitcoina, Bitcoin Cash i Stellara wskazują na zatrzymanie
optymalizatora poza dopuszczalnym obszarem. Jądro gaussowskie zależy od kwadratu szerokości, więc
oszacowanie pozostaje poprawne liczbowo, ale wartości tych nie należy czytać jako dobranej
gładkości.

## Zmienność krótkookresowa

Dla każdego szeregu oszacowano GARCH(1,1) oraz GJR-GARCH(1,1,1). Specyfikację asymetryczną
zachowano tylko tam, gdzie parametr dźwigni jest istotny na poziomie 5% i gdzie BIC jest niższy.

Warunek spełnia wyłącznie Ethereum. Dla pozostałych 11 szeregów brak podstaw do twierdzenia, że
ujemne szoki wpływają na przyszłą zmienność inaczej niż dodatnie.

In [ ]:
parameters = model.run()
parameters

In [ ]:
set_figures_dir(paths.FIGURES)

for name in data_load.CRYPTO_LIST:
    plot_garch_decomposition(name)

print(f'zapisano {len(data_load.CRYPTO_LIST)} wykresow do {paths.FIGURES.name}/')

## Test PSY

Statystyka GSADF to supremum statystyk ADF po wszystkich dopuszczalnych oknach, w których zmienia
się zarówno początek, jak i koniec. Sekwencja BSADF podaje to supremum dla każdego punktu
końcowego osobno i służy do datowania epizodów.

Rozkład zerowy generuje wild bootstrap na 10 000 ścieżek, zaburzanych dwupunktowym rozkładem
Mammena i odtwarzających strukturę zmienności szeregu. Ścieżki zawierają deterministyczny dryf
równy medianie historycznych przyrostów logarytmicznych cen, co oddziela trend wzrostowy od
eksplozji cenowej.

In [ ]:
print(f'sciezek bootstrapu: {validation.N_BOOTSTRAP}')
print(f'minimalne okno: {validation.MIN_WINDOW} obserwacji')
print(f'kwantyl kalibracji krzywej: {validation.CV_QUANTILE}')
print(f'poziom istotnosci: {validation.BUBBLE_LEVEL}')
print(f'ziarno generatora: {validation.SEED}')

In [ ]:
summary, episodes = validation.run()
summary

## Wyniki

Epizody wybuchowe wyznacza przekroczenie krzywej wartości krytycznych przez sekwencję BSADF.
Kolumna `meets_min_duration` mówi, czy epizod osiąga zalecaną minimalną długość `ln(T)`, która dla
tej próby wynosi 7 dni. Krótsze przekroczenia zostają w tabeli, zamiast być z niej usuwane.

In [ ]:
episodes[episodes['meets_min_duration']].reset_index(drop=True)

In [ ]:
bubbles = summary.loc[summary['bubble'], ['crypto', 'gsadf', 'p_value']]
print(f"banki na poziomie {validation.BUBBLE_LEVEL}: {len(bubbles)} z {len(summary)}")
bubbles.reset_index(drop=True)

Wykresy obu etapów leżą w `reports/figures/`, a tabele wynikowe w `reports/`. Zestawienie liczb i opis metody jest w
[README](../README.md).